# [UM]-Net — Directed Graph Agent Network

Mirrors `04_dg_agent_network.ipynb`.  What flows between agents is
**encodings, not HLLSets**: each [UM] node is an `HLLSetFilter`, and the
network converges without coordination because every node is an IICA morphism
and CRDT union is monotonic.  The streams here are quantized V-JEPA tids.

In [1]:
import sys
import pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "ewm_jepa").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from ewm_jepa import Quantizer
import hllset_py
from hllset_cortex import HLLSetFilter
from hllset_cortex.domain import encoding_tokenizer, hllset_from_ids, tid

# A deterministic tid-stream source: the same quantizer the JEPA pipeline
# uses, fed with synthetic encodings.  The EWM boundary is encoding-agnostic —
# these tids are interchangeable with V-JEPA quantized patch encodings.
def make_tid_stream(n=200, dim=1024, codebook_size=2048, seed=0):
    q = Quantizer(dim=dim, codebook_size=codebook_size, device="cpu", seed=seed)
    # decorrelate the data seed from the codebook seed so streams differ
    z = torch.randn(n, dim, generator=torch.Generator().manual_seed(1000 + seed))
    return q.ids_to_text(q.encode(z))


## 1. Agent definition

In [2]:
from dataclasses import dataclass, field

@dataclass
class UMAgent:
    name: str
    filt: object = field(default_factory=HLLSetFilter)

    def __post_init__(self):
        self.filt.tokenizer = encoding_tokenizer()

    def process(self, text):
        res = self.filt.process_text(text)
        return " ".join(t for t in res.token_strings if "\x00" not in t)

    @property
    def hllset(self):
        # fingerprint of everything this agent has seen
        top = hllset_py.HLLSet()
        # last result is kept in _last
        return getattr(self, "_last", top)

agents = {name: UMAgent(name) for name in ["a", "b", "c", "d", "e"]}
print("agents:", list(agents))


agents: ['a', 'b', 'c', 'd', 'e']


## 2. Fire-and-forget: encodings in, encodings out

In [3]:
streams = {
    "a": make_tid_stream(n=60, seed=10),
    "b": make_tid_stream(n=60, seed=11),
}
outs = {name: agents[name].process(text) for name, text in streams.items()}
for name, out in outs.items():
    print(f"agent {name}: {len(out.split())} tids out")


agent a: 55 tids out
agent b: 58 tids out


## 3. Diamond topology: fork → process → merge

Two upstream agents process different views; their encodings merge at a
confluence agent.  The merge is union — monotonic, commutative, and
convergent by construction (no consensus).

In [4]:
up_a = outs["a"]
up_b = outs["b"]
merged = up_a + " " + up_b
out_c = agents["c"].process(merged)
print("confluence agent c:", len(out_c.split()), "tids")


confluence agent c: 102 tids


## 4. Robustness: loss, duplicates, ordering

The same CRDT properties that make HLLSet union idempotent make the network
insensitive to message loss, duplication, and reordering.

In [5]:
base = streams["a"]
dup = base + " " + base                       # duplicate delivery
reordered = " ".join(base.split()[::-1])      # reordered delivery

h_base = hllset_py.HLLSet.from_tokens(base.split())
h_dup = hllset_py.HLLSet.from_tokens(dup.split())
h_re = hllset_py.HLLSet.from_tokens(reordered.split())
print("dup == base      :", h_dup.content_key() == h_base.content_key())
print("reordered == base:", h_re.content_key() == h_base.content_key())


dup == base      : True
reordered == base: True


## 5. [E] → Graph → [E] feedback loop

The graph output feeds back as the next input: the network becomes a
recurrent EWM structure — short-term memory through temporal separation,
exactly the [UM]-Net cycle of `main.tex` §4.4.

In [6]:
x = streams["a"]
for step in range(3):
    x = agents["d"].process(agents["e"].process(x))
    print(f"loop {step}: {len(x.split())} tids")


loop 0: 53 tids
loop 1: 53 tids
loop 2: 53 tids


## 6. Summary

The [UM]-Net over V-JEPA encodings needs no handshakes, no retries, and no
leader election.  IICA morphisms + monotonic union = convergence by
construction — the property that lets EWM scale to swarms of world-model
nodes.